In [10]:
import os
import cv2
import numpy as np

from tqdm import tqdm
from pathlib import Path
import pandas as pd
from matplotlib import pyplot as plt

In [11]:
Train_Img_Path = r"C:\Users\kiran\Downloads\archive\dataset-splitM\Training\images"
Train_Mask_Path = r"C:\Users\kiran\Downloads\archive\dataset-splitM\Training\GT"

Test_Img_Path = r"C:\Users\kiran\Downloads\archive\dataset-splitM\Testing\images"
Test_Mask_Path = r"C:\Users\kiran\Downloads\archive\dataset-splitM\Testing\GT"

Output_Dir = r"C:\Users\kiran\Downloads\Processed"
os.makedirs(Output_Dir, exist_ok=True)

In [26]:
Catagories = [
{  
    "id":0,
    "name": "person",
    "supercategory": "camouflage"
}
]

In [19]:
Min_componenet_Area = 500
def extract_componenets(mask):
    _,binary = cv2.threshold(mask,127,255,cv2.THRESH_BINARY)
    num_labels, labels, stats = cv2.connectedComponentsWithStats(binary)
    componets= []
    for i in range(1, num_labels):
        x,y,w,h,area = stats[i]
        if area < Min_componenet_Area:
            continue
        componet_mask = np.zeros_like(binary)
        componet_mask[labels == i] = 255

        componets.append({
            "bbox": [x,y,w,h],
            "area": area,
            "mask": componet_mask   
        })
    return componets


In [25]:
def mask_to_polygon(mask):

    contours,_ = cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    polygons =[]
    for contour in contours:
        contour = contour.flatten().tolist()

        if len(contour) >= 6:
            polygons.append(contour)
    
    return polygons

In [ ]:
def create_coco_annotations(image_dir, mask_dir,output_json):

    coco ={
    "images": [],
    "annotations": [],
    "catagories": Catagories 
    }
    
    annotation_id = 0, 
    image_id =0

    imgae_files = sorted(os.listdir(image_dir))
    

